# Fetching data at warp speed

In [3]:
# 1. Install dependencies
!pip install dukascopy-python pandas pyarrow joblib -q

import os
import time
import pandas as pd
import dukascopy_python as dp
from datetime import datetime
from pathlib import Path
from joblib import Parallel, delayed

# --- CONFIGURATION ---
# NOTE: If you want to keep files permanently, mount Google Drive:
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = Path("/content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ["AUD/USD", "NZD/USD"]
MONTHS = [
    "201901", "201902", "201903", "201904", "201905", "201906",
    "201907", "201908", "201909", "201910", "201911", "201912",
    "202001", "202002", "202003", "202004", "202005", "202006",
    "202007", "202008", "202009", "202010", "202011", "202012"]

# PARALLELISM: Use -1 to use all cores, or 4-8 to avoid getting rate-limited
N_JOBS = 4 

# --- HELPER FUNCTIONS ---
def get_month_range(yyyymm: str):
    year, month = int(yyyymm[:4]), int(yyyymm[4:6])
    start = datetime(year, month, 1)
    end = datetime(year + 1, 1, 1) if month == 12 else datetime(year, month + 1, 1)
    return start, end

def process_job(pair, yyyymm, compression="snappy", max_retries=3):
    """Function to be executed in parallel with existence check and retries"""
    symbol = pair.replace("/", "").lower()
    
    # 1. EXISTENCE CHECK: Define file paths first
    out_name_bid = f"{symbol}_dukascopy_bid_{yyyymm}.parquet"
    out_name_ask = f"{symbol}_dukascopy_ask_{yyyymm}.parquet"
    out_path_bid = OUT_DIR / out_name_bid
    out_path_ask = OUT_DIR / out_name_ask

    # If both files already exist, skip the download completely
    if out_path_bid.exists() and out_path_ask.exists():
        return f"⏭️ Skipped (Already Exists): {pair} {yyyymm}"

    start, end = get_month_range(yyyymm)
    
    # 2. RETRY LOGIC: Handle Dukascopy connection drops
    for attempt in range(max_retries):
        try:
            df = dp.fetch(
                instrument=pair,
                interval=dp.INTERVAL_TICK,
                offer_side=dp.OFFER_SIDE_BID, 
                start=start,
                end=end,
            )

            if df is None or len(df) == 0:
                return f"⚠️ Empty: {pair} {yyyymm}"

            results_stats = []
            for side, price_col, vol_col in [("bid", "bidPrice", "bidVolume"), ("ask", "askPrice", "askVolume")]:
                out_name = out_name_bid if side == "bid" else out_name_ask
                out_path = out_path_bid if side == "bid" else out_path_ask
                
                pd.DataFrame({
                    "datetime": df.index,
                    "symbol": symbol.upper(),
                    "price_type": side.upper(),
                    "price": df[price_col].values,
                    "volume": df[vol_col].values,
                }).to_parquet(out_path, engine="pyarrow", compression=compression, index=False)
                
                results_stats.append(f"{out_name} ({len(df):,} rows)")
                
            return f"✅ Success: {pair} {yyyymm} -> " + " & ".join(results_stats)
        
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(5) # Wait 5 seconds before retrying
                continue
            return f"❌ FAILED: {pair} {yyyymm} | Error after {max_retries} attempts: {str(e)}"

# --- EXECUTION ---
jobs = [(p, m) for p in PAIRS for m in MONTHS]
print(f"🚀 Starting Parallel Download of {len(jobs)} jobs using {N_JOBS} workers...\n")

t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="threading")(
    delayed(process_job)(p, m) for p, m in jobs
)

for r in results:
    print(r)

elapsed = time.time() - t0
print(f"\n✨ All done! Finished in {elapsed:.1f}s")
print(f"📁 Files saved to: {OUT_DIR.absolute()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚀 Starting Parallel Download of 48 jobs using 4 workers...



INFO:DUKASCRIPT:current timestamp :2019-03-01T12:41:48.166000
INFO:DUKASCRIPT:current timestamp :2019-01-02T07:08:51.763000
INFO:DUKASCRIPT:current timestamp :2019-04-01T10:48:29.093000
INFO:DUKASCRIPT:current timestamp :2019-02-01T09:59:30.328000
INFO:DUKASCRIPT:current timestamp :2019-03-01T21:53:10.538000
INFO:DUKASCRIPT:current timestamp :2019-01-02T11:55:15.472000
INFO:DUKASCRIPT:current timestamp :2019-04-02T00:12:43.017000
INFO:DUKASCRIPT:current timestamp :2019-02-01T17:46:16.615000
INFO:DUKASCRIPT:current timestamp :2019-03-04T08:20:06.512000
INFO:DUKASCRIPT:current timestamp :2019-01-02T18:37:52.170000
INFO:DUKASCRIPT:current timestamp :2019-04-02T10:04:10.051000
INFO:DUKASCRIPT:current timestamp :2019-02-04T08:46:43.326000
INFO:DUKASCRIPT:current timestamp :2019-01-03T01:54:33.797000
INFO:DUKASCRIPT:current timestamp :2019-04-02T22:28:32.506000
INFO:DUKASCRIPT:current timestamp :2019-03-04T16:20:31.170000
INFO:DUKASCRIPT:current timestamp :2019-02-04T21:52:46.137000
INFO:DUK

✅ Success: AUD/USD 201901 -> audusd_dukascopy_bid_201901.parquet (1,619,130 rows) & audusd_dukascopy_ask_201901.parquet (1,619,130 rows)
✅ Success: AUD/USD 201902 -> audusd_dukascopy_bid_201902.parquet (1,407,908 rows) & audusd_dukascopy_ask_201902.parquet (1,407,908 rows)
✅ Success: AUD/USD 201903 -> audusd_dukascopy_bid_201903.parquet (1,367,216 rows) & audusd_dukascopy_ask_201903.parquet (1,367,216 rows)
✅ Success: AUD/USD 201904 -> audusd_dukascopy_bid_201904.parquet (1,186,935 rows) & audusd_dukascopy_ask_201904.parquet (1,186,935 rows)
✅ Success: AUD/USD 201905 -> audusd_dukascopy_bid_201905.parquet (1,388,288 rows) & audusd_dukascopy_ask_201905.parquet (1,388,288 rows)
✅ Success: AUD/USD 201906 -> audusd_dukascopy_bid_201906.parquet (1,388,717 rows) & audusd_dukascopy_ask_201906.parquet (1,388,717 rows)
✅ Success: AUD/USD 201907 -> audusd_dukascopy_bid_201907.parquet (1,102,101 rows) & audusd_dukascopy_ask_201907.parquet (1,102,101 rows)
✅ Success: AUD/USD 201908 -> audusd_dukas